In [ ]:
# Optional Colab mount: uncomment if running on Colab.
# from google.colab import drive
# drive.mount("/content/drive")

import os

# Assumes you've %cd'd into the eButterfly working directory first, e.g.:
# %cd /content/drive/MyDrive/CISO/eButterfly/   # set WORK_DIR below to your local data dir
WORK_DIR_OVERRIDE = os.environ.get("WORK_DIR", os.getcwd())
os.chdir(WORK_DIR_OVERRIDE)
# All paths captured as absolute so they survive the os.chdir below.

WORK_DIR    = os.path.abspath('.')
CISO_DIR    = os.path.abspath('../CISO-SDM')

import subprocess
if not os.path.exists(CISO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/RolnickLab/CISO-SDM", CISO_DIR],
        check=True,
    )
    print("Cloned CISO-SDM to", CISO_DIR)
else:
    print("CISO-SDM already present, skipping clone.")
DATA_DIR    = WORK_DIR                                          # CSV + splits + prepped files
CKPT_DIR    = WORK_DIR                                          # checkpoints saved here
RESULTS_DIR = os.path.join(WORK_DIR, 'results_CISO_butterfly_100_epochs_seed1338')

os.makedirs(RESULTS_DIR, exist_ok=True)

# CISO expects to read from <CISO_DIR>/data — symlink it to our working dir.
# Absolute target so it remains valid after we chdir into CISO_DIR.
data_link = os.path.join(CISO_DIR, 'data')
if not os.path.exists(data_link):
    os.symlink(WORK_DIR, data_link)
    print(f"Symlinked {data_link} → {WORK_DIR}")
else:
    print("Symlink already exists, skipping.")

# Move into CISO repo so `!python main.py ...` resolves.
os.chdir(CISO_DIR)

!pip install "numpy<2.0"
!pip install -r requirements.txt

print("\nSetup complete ✓")
print(f"  cwd (now):     {os.getcwd()}")
print(f"  CISO repo:     {CISO_DIR}")
print(f"  work/data dir: {DATA_DIR}")
print(f"  ckpts dir:     {CKPT_DIR}")
print(f"  results dir:   {RESULTS_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/CISO/eButterfly
Symlink already exists, skipping.

Setup complete ✓
  cwd (now):     /content/drive/MyDrive/CISO/CISO-SDM
  CISO repo:     /content/drive/MyDrive/CISO/CISO-SDM
  work/data dir: /content/drive/MyDrive/CISO/eButterfly
  ckpts dir:     /content/drive/MyDrive/CISO/eButterfly
  results dir:   /content/drive/MyDrive/CISO/eButterfly/results_CISO_butterfly_100_epochs_seed1338


In [ ]:
import numpy as np
import pandas as pd
import json
import yaml
import shutil
from pathlib import Path

assert np.__version__.startswith("1."), f"numpy must be <2.0 for elapid; got {np.__version__}"
print(f"numpy {np.__version__}, pandas {pd.__version__}")
print("Imports OK")

numpy 1.26.4, pandas 2.3.0
Imports OK


Upload modified main.py with CSVLogger, not cometlogger

## 1. Load Your Data

In [ ]:
# ── Paths — edit these if your files are elsewhere ────────────────────────────
DATA_CSV    = f'{DATA_DIR}/ebutterfly_na_2011_2025.csv'
SPLITS_JSON = f'{DATA_DIR}/ebutterfly_splits.json'

butterflies = pd.read_csv(DATA_CSV)
butterflies = butterflies.reset_index(drop=True)   # ensure clean 0-based index

with open(SPLITS_JSON) as f:
    butterfly_splits = json.load(f)

print(f"Loaded {len(butterflies):,} rows × {len(butterflies.columns):,} columns")
print(f"Split keys: {list(butterfly_splits.keys())}")
butterflies.head(2)


Loaded 17,077 rows × 191 columns
Split keys: ['num_rows', 'meta', 'train', 'val', 'test']


,time,latitude,longitude,env_t2m_min,env_t2m_max,env_t2m_mean,env_tp_sum,env_ndvi,env_evi,env_soil_bdod,...,Tharsalea hyllus,Thymelicus lineola,Urbanus dorantes,Urbanus procne,Urbanus proteus,Vanessa atalanta,Vanessa cardui,Vanessa virginiensis,Vernia verna,Zerene cesonia
0,2025-05-16,44.481715,-77.682266,286.97748,299.25006,291.72070,5.594984e-03,0.6052,0.3350,118.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2015-04-19,34.046186,-118.275999,286.35144,299.09650,292.28778,5.420297e-07,0.1364,0.0748,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Identify column groups
env_cols     = [c for c in butterflies.columns if c.startswith('env_')]
species_cols = [c for c in butterflies.columns
                if c not in env_cols + ['time', 'latitude', 'longitude']]

# eButterfly daily set has no bioclim. Group env into:
#   - "worldclim_cols" (variable name kept for downstream compatibility):
#       ERA5 daily t2m/tp + MODIS NDVI/EVI  (dynamic)
#   - "soilgrid_cols": SoilGrids + DEM      (static)
soilgrid_cols  = [c for c in env_cols if c.startswith('env_soil') or c == 'env_dem']
worldclim_cols = [c for c in env_cols if c not in soilgrid_cols]

print(f"Climate/dynamic cols ({len(worldclim_cols)}): {worldclim_cols}")
print(f"Static cols          ({len(soilgrid_cols)}):  {soilgrid_cols}")
print(f"Species cols         ({len(species_cols)}):   {species_cols[:5]} ...")


Climate/dynamic cols (6): ['env_t2m_min', 'env_t2m_max', 'env_t2m_mean', 'env_tp_sum', 'env_ndvi', 'env_evi']
Static cols          (9):  ['env_soil_bdod', 'env_soil_cec', 'env_soil_cfvo', 'env_soil_clay', 'env_soil_nitrogen', 'env_soil_phh2o', 'env_soil_sand', 'env_soil_silt', 'env_dem']
Species cols         (173):   ['Abaeis nicippe', 'Aglais milberti', 'Amblyscirtes hegon', 'Amblyscirtes vialis', 'Anartia fatima'] ...


## 2. Build Split Indices

In [ ]:
# Inspect what split values look like
print("First 5 train values:", butterfly_splits['train'][:5])
print("Type:", type(butterfly_splits['train'][0]))
print(f"train={len(butterfly_splits['train']):,}  "
      f"val={len(butterfly_splits['val']):,}  "
      f"test={len(butterfly_splits['test']):,}")
print(f"Total: {len(butterfly_splits['train'])+len(butterfly_splits['val'])+len(butterfly_splits['test']):,}  "
      f"vs rows: {len(butterflies):,}")


First 5 train values: [0, 1, 2, 3, 4]
Type: <class 'int'>
train=13,825  val=950  test=2,302
Total: 17,077  vs rows: 17,077


In [ ]:
# Build numpy index arrays
train_split = np.array(butterfly_splits['train'])
val_split   = np.array(butterfly_splits['val'])
test_split  = np.array(butterfly_splits['test'])

print(f"train_split: {train_split.shape}, max={train_split.max()}")
print(f"val_split:   {val_split.shape},   max={val_split.max()}")
print(f"test_split:  {test_split.shape},  max={test_split.max()}")
assert train_split.max() < len(butterflies), "Train index out of bounds!"
assert val_split.max()   < len(butterflies), "Val index out of bounds!"
assert test_split.max()  < len(butterflies), "Test index out of bounds!"
print("✓ All indices in bounds")


train_split: (13825,), max=17076
val_split:   (950,),   max=17062
test_split:  (2302,),  max=17072
✓ All indices in bounds


## 3. Prepare Files in CISO's Expected Format

In [ ]:
# Identity rename — CISO reads whatever column names are listed in env_columns
# (config.yaml). Keep the original env_* names; CISO doesn't care what they're
# called as long as the CSV columns and the config list agree.
worldclim_rename = {c: c for c in worldclim_cols}

print("Climate/dynamic rename map (identity):")
for k, v in worldclim_rename.items():
    print(f"  {k} → {v}")


Climate/dynamic rename map (identity):
  env_t2m_min → env_t2m_min
  env_t2m_max → env_t2m_max
  env_t2m_mean → env_t2m_mean
  env_tp_sum → env_tp_sum
  env_ndvi → env_ndvi
  env_evi → env_evi


In [ ]:
# Identity rename for static cols
soilgrid_rename = {c: c for c in soilgrid_cols}

print("Static rename map (identity):")
for k, v in soilgrid_rename.items():
    print(f"  {k} → {v}")


Static rename map (identity):
  env_soil_bdod → env_soil_bdod
  env_soil_cec → env_soil_cec
  env_soil_cfvo → env_soil_cfvo
  env_soil_clay → env_soil_clay
  env_soil_nitrogen → env_soil_nitrogen
  env_soil_phh2o → env_soil_phh2o
  env_soil_sand → env_soil_sand
  env_soil_silt → env_soil_silt
  env_dem → env_dem


In [ ]:
OUT_DIR = Path("data/eButterfly")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Add row ID column
butterflies['PlotObservationID'] = butterflies.index

# ── Climate/dynamic CSV (CISO reads as worldclim) ────────────────────────────
butterflies[['PlotObservationID'] + worldclim_cols]\
    .rename(columns=worldclim_rename)\
    .to_csv(OUT_DIR / "worldclim_data.csv", index=False)

# ── Static (soil + dem) CSV (CISO reads as soilgrid) ─────────────────────────
butterflies[['PlotObservationID'] + soilgrid_cols]\
    .rename(columns=soilgrid_rename)\
    .to_csv(OUT_DIR / "soilgrid_data.csv", index=False)

# ── Filter species: >= 100 occurrences ───────────────────────────────────────
targets_full          = butterflies[species_cols].to_numpy().astype(np.float32)
species_counts        = targets_full.sum(axis=0)
keep                  = species_counts >= 100
targets               = targets_full[:, keep]
species_cols_filtered = [s for s, k in zip(species_cols, keep) if k]

print(f"Species before filtering: {len(species_cols):,}")
print(f"Species after  filtering: {len(species_cols_filtered):,}")

# ── Species list CSV ─────────────────────────────────────────────────────────
pd.DataFrame({'species': species_cols_filtered})\
    .to_csv(OUT_DIR / "species_merge_duplicates_v2.csv", index=False)

# ── Species occurrences .npy ─────────────────────────────────────────────────
np.save(OUT_DIR / "merged_species_occurrences_v2.npy", targets)

# ── Split indices .npy ───────────────────────────────────────────────────────
np.save(OUT_DIR / "train_indices.npy",      train_split)
np.save(OUT_DIR / "validation_indices.npy", val_split)
np.save(OUT_DIR / "test_indices.npy",       test_split)

print("\nFiles saved:")
for f in sorted(OUT_DIR.iterdir()):
    print(f"  {f.name}")


Species before filtering: 173
Species after  filtering: 171

Files saved:
  merged_species_occurrences_v2.npy
  soilgrid_data.csv
  species_merge_duplicates_v2.csv
  test_indices.npy
  train_indices.npy
  validation_indices.npy
  worldclim_data.csv


## 4. Verify Data Integrity

In [ ]:
wc  = pd.read_csv(OUT_DIR / "worldclim_data.csv")
sg  = pd.read_csv(OUT_DIR / "soilgrid_data.csv")
sp  = pd.read_csv(OUT_DIR / "species_merge_duplicates_v2.csv")
tgt = np.load(OUT_DIR / "merged_species_occurrences_v2.npy")
tr  = np.load(OUT_DIR / "train_indices.npy")
val = np.load(OUT_DIR / "validation_indices.npy")
te  = np.load(OUT_DIR / "test_indices.npy")

print(f"climate/dyn (wc) : {wc.shape}   cols: {wc.columns.tolist()}")
print(f"static (sg)      : {sg.shape}   cols: {sg.columns.tolist()}")
print(f"species          : {sp.shape}   cols: {sp.columns.tolist()}")
print(f"targets          : {tgt.shape}")
print(f"train/val/test   : {tr.shape} / {val.shape} / {te.shape}")

# Assertions
n_env = len(wc.columns) - 1 + len(sg.columns) - 1   # minus PlotObservationID
assert tgt.shape[1] == len(sp), \
    f"Species mismatch: targets has {tgt.shape[1]} but species list has {len(sp)}"
assert len(wc) == len(tgt), \
    f"Row mismatch: climate {len(wc)} vs targets {len(tgt)}"
assert tr.max() < len(tgt) and val.max() < len(tgt) and te.max() < len(tgt), \
    "Split index out of bounds!"

print(f"\n✓ All checks passed")
print(f"  input_dim   (for config) = {n_env}")
print(f"  num_classes (for config) = {tgt.shape[1]}")


climate/dyn (wc) : (17077, 7)   cols: ['PlotObservationID', 'env_t2m_min', 'env_t2m_max', 'env_t2m_mean', 'env_tp_sum', 'env_ndvi', 'env_evi']
static (sg)      : (17077, 10)   cols: ['PlotObservationID', 'env_soil_bdod', 'env_soil_cec', 'env_soil_cfvo', 'env_soil_clay', 'env_soil_nitrogen', 'env_soil_phh2o', 'env_soil_sand', 'env_soil_silt', 'env_dem']
species          : (171, 1)   cols: ['species']
targets          : (17077, 171)
train/val/test   : (13825,) / (950,) / (2302,)

✓ All checks passed
  input_dim   (for config) = 15
  num_classes (for config) = 171


## 5. Write Config File

In [ ]:
# Build the env_columns list from the renamed columns (excluding PlotObservationID)
env_columns_for_config = (
    [worldclim_rename[c] for c in worldclim_cols] +
    [soilgrid_rename[c]  for c in soilgrid_cols]
)
print(f"env_columns ({len(env_columns_for_config)}):")
print(env_columns_for_config)

env_columns (15):
['env_t2m_min', 'env_t2m_max', 'env_t2m_mean', 'env_tp_sum', 'env_ndvi', 'env_evi', 'env_soil_bdod', 'env_soil_cec', 'env_soil_cfvo', 'env_soil_clay', 'env_soil_nitrogen', 'env_soil_phh2o', 'env_soil_sand', 'env_soil_silt', 'env_dem']


In [ ]:
CONFIG_PATH = Path("configs/config_ciso_mydata.yaml")
CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)

# ── Edit these as needed ──────────────────────────────────────────────────────
EXPERIMENT_NAME = "butterfly_ciso_mydata"
CHECKPOINT_DIR  = CKPT_DIR
MAX_EPOCHS      = 100
BATCH_SIZE      = 64
LEARNING_RATE   = 1e-3
# ─────────────────────────────────────────────────────────────────────────────

config_dict = {
    'mode': 'train',
    'dataset_name': 'sPlot',
    'model': {
        'name': 'CISOModel',
        'input_dim': n_env,
        'hidden_dim': 256,
        'num_classes': int(tgt.shape[1]),
        'backbone': 'SimpleMLPBackbone',
    },
    'training': {
        'seed': 1338,
        'learning_rate': LEARNING_RATE,
        'max_epochs': MAX_EPOCHS,
        'accelerator': 'gpu',
        'devices': 1,
    },
    'logger': {
        'project_name': 'eButterfly',
        'experiment_name': EXPERIMENT_NAME,
        'experiment_key': '',
        'checkpoint_path': CHECKPOINT_DIR,
        'checkpoint_name': '',
        'save_preds_path': '',
    },
    'data': {
        'dataloader_to_use': 'sPlotMaskedDataset',
        'base': 'data/eButterfly',
        'train': 'train_indices.npy',
        'validation': 'validation_indices.npy',
        'test': 'test_indices.npy',
        'targets': 'merged_species_occurrences_v2.npy',
        'worldclim_data_path': 'worldclim_data.csv',
        'soilgrid_data_path':  'soilgrid_data.csv',
        'species_list': 'species_merge_duplicates_v2.csv',
        'species_occurrences_threshold': 100,
        'batch_size': BATCH_SIZE,
        'env_columns': env_columns_for_config,
        'partial_labels': {
            'use': True,
            'quantized_mask_bins': 1,      # binary presence/absence
            'train_known_ratio': 0.75,     # max 75% species known during training
            'eval_known_ratio': 0,         # fully unconditioned at eval
            'predict_family_of_species': -1,  # -1 = predict all species
        },
    },
}

with open(CONFIG_PATH, 'w') as f:
    yaml.dump(config_dict, f, default_flow_style=False, sort_keys=False)

print(f"Config written to: {CONFIG_PATH}")
print()
print(open(CONFIG_PATH).read())


Config written to: configs/config_ciso_mydata.yaml

mode: train
dataset_name: sPlot
model:
  name: CISOModel
  input_dim: 15
  hidden_dim: 256
  num_classes: 171
  backbone: SimpleMLPBackbone
training:
  seed: 1338
  learning_rate: 0.001
  max_epochs: 100
  accelerator: gpu
  devices: 1
logger:
  project_name: eButterfly
  experiment_name: butterfly_ciso_mydata
  experiment_key: ''
  checkpoint_path: /content/drive/MyDrive/CISO/eButterfly
  checkpoint_name: ''
  save_preds_path: ''
data:
  dataloader_to_use: sPlotMaskedDataset
  base: data/eButterfly
  train: train_indices.npy
  validation: validation_indices.npy
  test: test_indices.npy
  targets: merged_species_occurrences_v2.npy
  worldclim_data_path: worldclim_data.csv
  soilgrid_data_path: soilgrid_data.csv
  species_list: species_merge_duplicates_v2.csv
  species_occurrences_threshold: 100
  batch_size: 64
  env_columns:
  - env_t2m_min
  - env_t2m_max
  - env_t2m_mean
  - env_tp_sum
  - env_ndvi
  - env_evi
  - env_soil_bdod
  -

## 6. Train

In [ ]:
import os

# CometLogger in CISO's main.py is instantiated with api_key=os.environ.get("COMET_API_KEY").
# If unset, Lightning raises "requires either api_key or save_dir".
# Set a dummy key and force offline mode so it never contacts the server.
os.environ["COMET_API_KEY"] = "offline-dummy"
os.environ["COMET_MODE"] = "OFFLINE"
os.environ.setdefault("COMET_OFFLINE_DIRECTORY", "comet_logs")

os.makedirs("comet_logs", exist_ok=True)

print("Comet env set: OFFLINE mode, dummy api_key, logs → ./comet_logs/")

Comet env set: OFFLINE mode, dummy api_key, logs → ./comet_logs/


In [ ]:
!python main.py --config configs/config_ciso_mydata.yaml

/content/drive/MyDrive/CISO/CISO-SDM/configs/config_ciso_mydata.yaml
Mode: train
Model Config: name='CISOModel' input_dim=15 hidden_dim=256 num_classes=171 backbone='SimpleMLPBackbone'
Data Path Config: dataloader_to_use='sPlotMaskedDataset' base='data/eButterfly' maxent_transform=False train='train_indices.npy' validation='validation_indices.npy' test='test_indices.npy' targets='merged_species_occurrences_v2.npy' worldclim_data_path='worldclim_data.csv' soilgrid_data_path='soilgrid_data.csv' species_occurrences_threshold=100 batch_size=64 species_list='species_merge_duplicates_v2.csv' env_columns=['env_t2m_min', 'env_t2m_max', 'env_t2m_mean', 'env_tp_sum', 'env_ndvi', 'env_evi', 'env_soil_bdod', 'env_soil_cec', 'env_soil_cfvo', 'env_soil_clay', 'env_soil_nitrogen', 'env_soil_phh2o', 'env_soil_sand', 'env_soil_silt', 'env_dem'] partial_labels=PartialLabels(use=True, predict_family_of_species=-1, train_known_ratio=0.75, eval_known_ratio=0.0, quantized_mask_bins=1)
Training Config: seed=

In [ ]:
# Check what checkpoint was saved
ckpt_dir = Path(CHECKPOINT_DIR)
if ckpt_dir.exists():
    ckpts = list(ckpt_dir.glob("**/*.ckpt")) + list(ckpt_dir.glob("**/*.pt"))
    print("Saved checkpoints:")
    for c in ckpts:
        print(f"  {c}")
else:
    print(f"Checkpoint dir not found: {ckpt_dir}")

Saved checkpoints:
  /content/drive/MyDrive/CISO/eButterfly/butterfly_ciso_mydata/1337/epoch=11-step=2604.ckpt
  /content/drive/MyDrive/CISO/eButterfly/butterfly_ciso_mydata/1337/last.ckpt
  /content/drive/MyDrive/CISO/eButterfly/butterfly_ciso_mydata/1337/epoch=29-step=6510.ckpt
  /content/drive/MyDrive/CISO/eButterfly/butterfly_ciso_mydata/1337/last-v1.ckpt
  /content/drive/MyDrive/CISO/eButterfly/butterfly_ciso_mydata/1338/epoch=33-step=7378.ckpt
  /content/drive/MyDrive/CISO/eButterfly/butterfly_ciso_mydata/1338/last.ckpt
  /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1337/epoch_17.pt
  /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1337/epoch_24.pt
  /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1337/epoch_31.pt
  /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1337/epoch_32.pt
  /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1337/epoch_33.pt
  /content/drive/MyDrive/CISO/eButterfly/masksdm_butterfly_1337/epoch_34.pt
  /content/d

## 7. Benchmark at Multiple Masking Levels

Evaluate CISO at `eval_known_ratio` ∈ {0.0, 0.25, 0.5, 0.75, 1.0}, corresponding to masking fractions p ∈ {1.0, 0.75, 0.5, 0.25, 0.0} — matching the four masking levels reported in STEM-LM.

In [ ]:
EVAL_RATIOS = [0.0, 0.25, 0.5, 0.75]

# Auto-detect checkpoint
CHECKPOINT_NAME = ""
ckpts = list(Path(CKPT_DIR).glob("**/*.ckpt"))
best = [p for p in ckpts if 'last' not in p.name]
if best:
    CHECKPOINT_NAME = str(sorted(best)[-1])
elif ckpts:
    CHECKPOINT_NAME = str(sorted(ckpts)[-1])

if CHECKPOINT_NAME:
    print(f"Auto-detected checkpoint: {CHECKPOINT_NAME}")
else:
    print("No checkpoint found — run training first")


Auto-detected checkpoint: /content/drive/MyDrive/CISO/eButterfly/butterfly_ciso_mydata/1338/epoch=33-step=7378.ckpt


In [ ]:
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

for ratio in EVAL_RATIOS:
    print(f"\n{'='*60}")
    print(f"Evaluating eval_known_ratio={ratio}  (mask p={1-ratio:.2f})")
    print('='*60, flush=True)

    ratio_str = str(ratio).replace('.', 'p')

    # Create a directory for this ratio's prediction shards
    preds_dir = f'{RESULTS_DIR}/preds_ratio_{ratio_str}'
    Path(preds_dir).mkdir(parents=True, exist_ok=True)   # ← fix

    test_config = config_dict.copy()
    test_config['mode'] = 'test'
    test_config['logger'] = config_dict['logger'].copy()
    test_config['logger']['checkpoint_name'] = CHECKPOINT_NAME
    test_config['logger']['save_preds_path'] = preds_dir  # ← directory, not .npy file
    test_config['data'] = config_dict['data'].copy()
    test_config['data']['partial_labels'] = config_dict['data']['partial_labels'].copy()
    test_config['data']['partial_labels']['eval_known_ratio'] = ratio

    cfg_path = f"configs/config_ciso_test_{ratio_str}.yaml"
    res_csv  = f"{RESULTS_DIR}/results_ratio_{ratio_str}.csv"

    with open(cfg_path, 'w') as f:
        yaml.dump(test_config, f, default_flow_style=False, sort_keys=False)

    !python main.py --config {cfg_path} --results_file_name {res_csv}

print("\nDone — all masking levels evaluated.")


Evaluating eval_known_ratio=0.0  (mask p=1.00)
/content/drive/MyDrive/CISO/CISO-SDM/configs/config_ciso_test_0p0.yaml
Mode: test
Model Config: name='CISOModel' input_dim=15 hidden_dim=256 num_classes=171 backbone='SimpleMLPBackbone'
Data Path Config: dataloader_to_use='sPlotMaskedDataset' base='data/eButterfly' maxent_transform=False train='train_indices.npy' validation='validation_indices.npy' test='test_indices.npy' targets='merged_species_occurrences_v2.npy' worldclim_data_path='worldclim_data.csv' soilgrid_data_path='soilgrid_data.csv' species_occurrences_threshold=100 batch_size=64 species_list='species_merge_duplicates_v2.csv' env_columns=['env_t2m_min', 'env_t2m_max', 'env_t2m_mean', 'env_tp_sum', 'env_ndvi', 'env_evi', 'env_soil_bdod', 'env_soil_cec', 'env_soil_cfvo', 'env_soil_clay', 'env_soil_nitrogen', 'env_soil_phh2o', 'env_soil_sand', 'env_soil_silt', 'env_dem'] partial_labels=PartialLabels(use=True, predict_family_of_species=-1, train_known_ratio=0.75, eval_known_ratio=0

## 8. Summarise Results

In [ ]:
# Compute the full STEM-LM metric set on CISO's test predictions.
# Matches STEMLM_metric.compute_per_species_metrics exactly:
#   - Per species s, only rows where s was MASKED contribute to that species' AUROC/AUPRC/CBI/Brier/ECE.
#     This is the analogue of STEM-LM's `labels != -100` filter and prevents leakage from
#     species shown to the model as known input.
#   - Mean + AUROC q25/q50/q75 across species.
#
# We re-run inference in-notebook so we can capture per-row MASKS alongside
# probs/targets (CISO's main.py only saves probs+targets per hotspot).
import os, yaml, json
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import spearmanr

from src.config import Config
from src.dataloaders.splot_dataloader import sPlotDataModule
from src.trainers.splot_trainer import sPlotTrainer

# ── Vendored from STEMLM_metric.py (verbatim) ───────────────────────────
def _safe_auc_roc(y, p):
    if y.size == 0 or len(set(y.tolist())) < 2 or np.isnan(p).any(): return float('nan')
    try: return float(roc_auc_score(y, p))
    except Exception: return float('nan')

def _safe_auc_pr(y, p):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    try: return float(average_precision_score(y, p))
    except Exception: return float('nan')

def _safe_brier(y, p):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    return float(np.mean((p - y.astype(np.float64))**2))

def _safe_ece(y, p, n_bins=15):
    if y.size == 0 or np.isnan(p).any(): return float('nan')
    edges = np.linspace(0, 1, n_bins+1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, n_bins-1)
    err = 0.0; n = p.size
    for b in range(n_bins):
        m = idx == b
        if not m.any(): continue
        err += (m.sum()/n) * abs(y[m].mean() - p[m].mean())
    return float(err)

def _safe_cbi(y, p, n_windows=101, width=0.1, min_per_window=10):
    if y.size == 0 or y.sum() == 0 or y.sum() == y.size or np.isnan(p).any(): return float('nan')
    pres = p[y==1]; bg = p[y==0]
    if pres.size == 0 or bg.size == 0: return float('nan')
    centers = np.linspace(0, 1, n_windows); half = width/2
    pe = np.full(n_windows, np.nan)
    for i, c in enumerate(centers):
        lo, hi = c-half, c+half
        n_bg = int(((bg>=lo)&(bg<=hi)).sum())
        if n_bg < min_per_window: continue
        e = n_bg/bg.size
        if e == 0: continue
        pe[i] = ((pres>=lo)&(pres<=hi)).sum()/pres.size / e
    ok = np.isfinite(pe)
    if ok.sum() < 3 or np.unique(pe[ok]).size < 2: return float('nan')
    try:
        rho = spearmanr(centers[ok], pe[ok]).statistic
        return float(rho) if np.isfinite(rho) else float('nan')
    except Exception: return float('nan')


def _per_species_metrics(probs, targets, masks):
    """Per-species metrics, FILTERED to rows where each species was masked.
    probs, targets, masks: (N, S) float / int / int. mask == -1 => masked target."""
    S = probs.shape[1]
    out = {k: {} for k in ['auc_roc','auc_pr','cbi','brier','ece']}
    n_kept_per_sp = []
    for s in range(S):
        keep = masks[:, s] == -1
        n_kept_per_sp.append(int(keep.sum()))
        if keep.sum() == 0:
            continue
        y = targets[keep, s].astype(np.int64)
        p = probs[keep, s].astype(np.float64)
        if y.sum() == 0 or y.sum() == y.size:
            continue
        out['auc_roc'][s] = _safe_auc_roc(y, p)
        out['auc_pr'][s]  = _safe_auc_pr(y, p)
        out['cbi'][s]     = _safe_cbi(y, p)
        out['brier'][s]   = _safe_brier(y, p)
        out['ece'][s]     = _safe_ece(y, p)
    return out, n_kept_per_sp

def _summarize(per_sp):
    def cl(d): return [v for v in d.values() if np.isfinite(v)]
    aucs = cl(per_sp['auc_roc']); prs = cl(per_sp['auc_pr'])
    cbis = cl(per_sp['cbi']);     bri = cl(per_sp['brier']);  ece = cl(per_sp['ece'])
    q = lambda x, p: float(np.quantile(x, p)) if x else float('nan')
    return {
        'mean_auc_roc': float(np.mean(aucs)) if aucs else float('nan'),
        'auc_roc_q25': q(aucs, .25), 'auc_roc_q50': q(aucs, .50), 'auc_roc_q75': q(aucs, .75),
        'mean_auc_pr': float(np.mean(prs)) if prs else float('nan'),
        'mean_cbi':    float(np.mean(cbis)) if cbis else float('nan'),
        'mean_brier':  float(np.mean(bri)) if bri else float('nan'),
        'mean_ece':    float(np.mean(ece)) if ece else float('nan'),
        'n_species':   len(aucs),
    }


def _get_seed(run_id, seed): return (run_id * (seed + (run_id - 1))) % (2**31 - 1)

def _run_inference_with_masks(test_cfg_path):
    """Load CISO checkpoint and run on test loader; return (probs, targets, masks) numpy.
    GPU-enabled.
    """
    cfg = Config(**yaml.safe_load(open(test_cfg_path)))
    seed = _get_seed(1, cfg.training.seed)

    ckpt = os.path.join(
        cfg.logger.checkpoint_path,
        cfg.logger.experiment_name,
        str(seed),
        cfg.logger.checkpoint_name,
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    dm = sPlotDataModule(cfg.data)
    dm.setup()

    task = sPlotTrainer(cfg).to(device)
    task.load_state_dict(torch.load(ckpt, map_location=device)["state_dict"])
    task.eval()

    probs_all, tgt_all, mask_all = [], [], []

    with torch.no_grad():
        for batch in dm.test_dataloader(num_workers=0, persistent_workers=False):
            # Move tensor leaves of the batch dict to GPU.
            batch_dev = {
                k: (v.to(device) if torch.is_tensor(v) else v)
                for k, v in batch.items()
            }

            logits = task(batch_dev)

            probs_all.append(torch.sigmoid(logits).cpu().numpy())
            tgt_all.append(batch_dev["targets"].cpu().numpy())
            mask_all.append(batch_dev["mask"].cpu().numpy())  # -1 = masked, 0/1 = known

    return (
        np.concatenate(probs_all, axis=0),
        np.concatenate(tgt_all, axis=0).astype(np.int64),
        np.concatenate(mask_all, axis=0).astype(np.int64),
    )


# ── Sweep: full STEM-LM metric set per eval_known_ratio ─────────────────
rows = []
for ratio in EVAL_RATIOS:
    rs = str(ratio).replace('.', 'p')
    cfg_path = f"configs/config_ciso_test_{rs}.yaml"
    if not Path(cfg_path).exists():
        print(f"missing {cfg_path} — skipping"); continue
    print(f"\n→ ratio={ratio} (mask p={1-ratio:.2f})", flush=True)
    probs, tgt, mask = _run_inference_with_masks(cfg_path)
    per_sp, n_kept = _per_species_metrics(probs, tgt, mask)
    summ = _summarize(per_sp)
    summ.update({'eval_known_ratio': ratio, 'masking_p': round(1-ratio, 2),
                 'avg_masked_rows_per_sp': float(np.mean(n_kept))})
    rows.append(summ)
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in summ.items()})

summary = pd.DataFrame(rows).set_index('masking_p').sort_index()
cols = ['mean_auc_roc','auc_roc_q25','auc_roc_q50','auc_roc_q75',
        'mean_auc_pr','mean_cbi','mean_brier','mean_ece','n_species','avg_masked_rows_per_sp']
summary = summary[['eval_known_ratio'] + cols]
print("\n── CISO benchmark (STEM-LM-faithful, masked-only filter) ─────────")
print(summary.to_string())
out_csv = f"{RESULTS_DIR}/ciso_benchmark_summary.csv"
summary.to_csv(out_csv)
print(f"\nSaved to {out_csv}")



→ ratio=0.0 (mask p=1.00)
Number of classes: 171


/tmp/ipykernel_24850/4080895298.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)["state_dict"])


{'mean_auc_roc': 0.8101, 'auc_roc_q25': 0.7197, 'auc_roc_q50': 0.8253, 'auc_roc_q75': 0.9336, 'mean_auc_pr': 0.1553, 'mean_cbi': 0.2987, 'mean_brier': 0.0289, 'mean_ece': 0.0168, 'n_species': 151, 'eval_known_ratio': 0.0, 'masking_p': 1.0, 'avg_masked_rows_per_sp': 2302.0}

→ ratio=0.25 (mask p=0.75)
Number of classes: 171


/tmp/ipykernel_24850/4080895298.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)["state_dict"])


{'mean_auc_roc': 0.8311, 'auc_roc_q25': 0.7522, 'auc_roc_q50': 0.8495, 'auc_roc_q75': 0.9428, 'mean_auc_pr': 0.1944, 'mean_cbi': 0.5027, 'mean_brier': 0.028, 'mean_ece': 0.0159, 'n_species': 151, 'eval_known_ratio': 0.25, 'masking_p': 0.75, 'avg_masked_rows_per_sp': 2017.3099}

→ ratio=0.5 (mask p=0.50)
Number of classes: 171


/tmp/ipykernel_24850/4080895298.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)["state_dict"])


{'mean_auc_roc': 0.8454, 'auc_roc_q25': 0.7759, 'auc_roc_q50': 0.8566, 'auc_roc_q75': 0.9464, 'mean_auc_pr': 0.2155, 'mean_cbi': 0.5583, 'mean_brier': 0.0278, 'mean_ece': 0.0156, 'n_species': 151, 'eval_known_ratio': 0.5, 'masking_p': 0.5, 'avg_masked_rows_per_sp': 1735.3743}

→ ratio=0.75 (mask p=0.25)
Number of classes: 171


/tmp/ipykernel_24850/4080895298.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  task.load_state_dict(torch.load(ckpt, map_location=device)["state_dict"])


{'mean_auc_roc': 0.8446, 'auc_roc_q25': 0.766, 'auc_roc_q50': 0.863, 'auc_roc_q75': 0.9426, 'mean_auc_pr': 0.2177, 'mean_cbi': 0.573, 'mean_brier': 0.0285, 'mean_ece': 0.0166, 'n_species': 146, 'eval_known_ratio': 0.75, 'masking_p': 0.25, 'avg_masked_rows_per_sp': 1418.6316}

── CISO benchmark (STEM-LM-faithful, masked-only filter) ─────────
           eval_known_ratio  mean_auc_roc  auc_roc_q25  auc_roc_q50  auc_roc_q75  mean_auc_pr  mean_cbi  mean_brier  mean_ece  n_species  avg_masked_rows_per_sp
masking_p                                                                                                                                                       
0.25                   0.75      0.844649     0.766004     0.862983     0.942610     0.217729  0.572987    0.028478  0.016565        146             1418.631579
0.50                   0.50      0.845436     0.775898     0.856558     0.946363     0.215502  0.558288    0.027831  0.015607        151             1735.374269
0.75        